# Aircraft Damage Detection + Part Risk Prediction

**Part 1:** YOLOv8 classifier for crack vs dent detection
**Part 2:** Rule-based part risk predictor (no historical data needed)

---
### Setup
Before running, rename the `valid` folder to `val`:
```
mv aircraft_damage_dataset_v1/valid aircraft_damage_dataset_v1/val
```
Ultralytics classification mode expects exactly `train/`, `val/`, `test/`.

In [1]:
# Install dependencies (run once)
!pip install ultralytics -q


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Part 1a — Train the classifier

In [2]:
from ultralytics import YOLO

# Path to dataset ROOT folder (the one containing train/val/test)
DATA_DIR = "aircraft_damage_dataset_v1"

# yolov8n-cls.pt = nano classifier, fastest, good for a first demo.
# Swap to yolov8s-cls.pt or yolov8m-cls.pt for higher accuracy later.
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=DATA_DIR,
    epochs=50,
    imgsz=224,
    batch=16,
    patience=10,          # early stop if val accuracy plateaus
    project="runs_damage",
    name="crack_dent_cls",
    pretrained=True,
)

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\sujan\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.115 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.12.1 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=aircraft_damage_dataset_v1, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, flip

In [3]:
# Evaluate on the held-out test set
metrics = model.val(data=DATA_DIR, split="test")
print("Test set metrics:", metrics.results_dict)

BEST_WEIGHTS = "runs_damage/crack_dent_cls/weights/best.pt"
print("Best model saved at:", BEST_WEIGHTS)

Ultralytics 8.4.31  Python-3.12.1 torch-2.11.0+cpu CPU (11th Gen Intel Core i7-1165G7 @ 2.80GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\sujan\OneDrive\Desktop\Aero_project\aircraft_damage_dataset_v1\train... found 300 images in 2 classes  
val: C:\Users\sujan\OneDrive\Desktop\Aero_project\aircraft_damage_dataset_v1\val... found 96 images in 2 classes  
test: C:\Users\sujan\OneDrive\Desktop\Aero_project\aircraft_damage_dataset_v1\test... found 50 images in 2 classes  
test: Fast image access  (ping: 0.30.1 ms, read: 3.61.4 MB/s, size: 52.9 KB)
test: Scanning C:\Users\sujan\OneDrive\Desktop\Aero_project\aircraft_damage_dataset_v1\test... 50 images, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 501.1it/s 0.1s
test: New cache created: C:\Users\sujan\OneDrive\Desktop\Aero_project\aircraft_damage_dataset_v1\test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 3.2it/s 1.3s0.6s
                   all       0.82      

## Part 1b — Run predictions (use this for your demo)

In [4]:
def predict_damage(image_path, weights_path="runs_damage/crack_dent_cls/weights/best.pt"):
    model = YOLO(weights_path)
    results = model(image_path)

    for r in results:
        probs = r.probs
        top1_label = r.names[probs.top1]
        top1_conf = probs.top1conf.item()

        print(f"Image: {image_path}")
        print(f"Predicted: {top1_label}  (confidence: {top1_conf:.2%})\n")
        print("All class probabilities:")
        for i, name in r.names.items():
            print(f"  {name}: {probs.data[i].item():.2%}")

    return results

In [ ]:
# Example usage — replace with a real image path
# predict_damage("path/to/your/image.jpg")

## Part 2 — Rule-based part risk predictor

No historical failure/maintenance dataset exists yet, so this uses
aviation engineering heuristics (age, cycle count proxy, past damage type)
as a transparent scoring system instead of an ML model. Swap this out for
a trained model once you're logging real outcomes.

In [5]:
from dataclasses import dataclass, field
from typing import List, Dict

# Maps components to base risk factors. Extend with real engineering input
# (MRO manuals, FAA Service Difficulty Reports, manufacturer AD/SB bulletins)
# as you get access to more domain knowledge.
PART_RISK_RULES: Dict[str, Dict] = {
    "wing_skin":        {"age_sensitivity": 0.8, "cycle_sensitivity": 1.0, "base_risk": 0.2},
    "fuselage_panel":   {"age_sensitivity": 0.7, "cycle_sensitivity": 0.9, "base_risk": 0.2},
    "landing_gear":     {"age_sensitivity": 0.5, "cycle_sensitivity": 1.2, "base_risk": 0.3},
    "engine_nacelle":   {"age_sensitivity": 1.0, "cycle_sensitivity": 0.6, "base_risk": 0.25},
    "tail_stabilizer":  {"age_sensitivity": 0.6, "cycle_sensitivity": 0.7, "base_risk": 0.15},
    "wing_stringer":    {"age_sensitivity": 0.9, "cycle_sensitivity": 1.1, "base_risk": 0.2},
    "cabin_door_frame": {"age_sensitivity": 0.5, "cycle_sensitivity": 0.8, "base_risk": 0.2},
}

# Which parts a given past-damage type tends to recur near / propagate to
DAMAGE_TYPE_LINKS: Dict[str, List[str]] = {
    "crack": ["wing_skin", "wing_stringer", "fuselage_panel", "tail_stabilizer"],
    "dent":  ["fuselage_panel", "cabin_door_frame", "engine_nacelle"],
}

@dataclass
class AircraftProfile:
    model: str
    year_of_manufacture: int
    current_year: int
    num_malfunctions: int
    past_damages: List[str] = field(default_factory=list)  # e.g. ["crack", "dent"]

def predict_risk(profile: AircraftProfile, top_n: int = 5) -> List[Dict]:
    age = max(profile.current_year - profile.year_of_manufacture, 0)

    # Rough proxy for cycle count from age + malfunction frequency.
    # Replace with real flight-hour/cycle data if the company provides it.
    est_cycles_factor = age * 1.0 + profile.num_malfunctions * 0.5

    scores = []
    for part, rule in PART_RISK_RULES.items():
        score = rule["base_risk"]
        score += rule["age_sensitivity"] * (age / 25)          # normalize ~25yr fleet life
        score += rule["cycle_sensitivity"] * (est_cycles_factor / 50)

        for damage in profile.past_damages:
            if part in DAMAGE_TYPE_LINKS.get(damage, []):
                score += 0.3

        scores.append({"part": part, "risk_score": round(min(score, 1.0), 3)})

    scores.sort(key=lambda x: x["risk_score"], reverse=True)
    return scores[:top_n]

In [6]:
# Example usage — swap in real inputs
profile = AircraftProfile(
    model="Boeing 737-800",
    year_of_manufacture=2005,
    current_year=2026,
    num_malfunctions=6,
    past_damages=["crack", "dent"],
)

ranked_parts = predict_risk(profile)

print(f"Aircraft: {profile.model} (age {profile.current_year - profile.year_of_manufacture} yrs)")
print("Top at-risk parts:\n")
for r in ranked_parts:
    print(f"  {r['part']:<18} risk score: {r['risk_score']}")

Aircraft: Boeing 737-800 (age 21 yrs)
Top at-risk parts:

  wing_skin          risk score: 1.0
  fuselage_panel     risk score: 1.0
  landing_gear       risk score: 1.0
  engine_nacelle     risk score: 1.0
  tail_stabilizer    risk score: 1.0
